# Lecture 20: t-tests — Supplementary Notebook
**Ling 250/450: Data Science for Linguistics** | Spring 2026

## Overview

This notebook accompanies the lecture slides on t-tests. Rather than
re-explaining the concepts from the slides, it focuses on the R mechanics:
working through the calculations by hand, running t-tests with `t.test()`,
and applying them to real vowel data.

Sections:

1. **The t-distribution**: visualizing how it differs from the Normal
2. **The z-test by hand**: working through the calculation to build intuition
3. **The one-sample t-test**: from scratch, then with `t.test()`
4. **The independent samples t-test**: comparing two groups in the vowels data

In [ ]:
vowels = read.csv("../data/vowels.csv", fileEncoding = "UTF-8-BOM")
library(dplyr)

---

## 1. The t-distribution

The t-distribution looks like a Normal distribution, but with **fatter tails**.
How fat depends on the **degrees of freedom** (df), which for a one-sample
t-test equals $N - 1$.

- **Low df** (small sample): much fatter tails — the test is more conservative
  because we're less certain about our estimate of $\sigma$
- **High df** (large sample): approaches the Normal distribution

R's `dt()` function gives the t-distribution density, analogous to `dnorm()`:

In [ ]:
x_vals = seq(-4, 4, by = 0.05)
normal_density = dnorm(x_vals, mean = 0, sd = 1)

par(mfrow = c(1, 2))

# df = 2: very fat tails
plot(x_vals, normal_density, type = "l", lty = 2, lwd = 2, col = "gray40",
     xlab = "value of t-statistic", ylab = "density", main = "df = 2",
     ylim = c(0, 0.45))
lines(x_vals, dt(x_vals, df = 2), lwd = 2, col = "black")
legend("topright", legend = c("t (df=2)", "Normal"),
       lty = c(1, 2), col = c("black", "gray40"), lwd = 2, cex = 0.8)

# df = 10: much closer to Normal
plot(x_vals, normal_density, type = "l", lty = 2, lwd = 2, col = "gray40",
     xlab = "value of t-statistic", ylab = "density", main = "df = 10",
     ylim = c(0, 0.45))
lines(x_vals, dt(x_vals, df = 10), lwd = 2, col = "black")
legend("topright", legend = c("t (df=10)", "Normal"),
       lty = c(1, 2), col = c("black", "gray40"), lwd = 2, cex = 0.8)

par(mfrow = c(1, 1))

The fatter tails mean the critical region starts **further out** for small
samples. Let's compare the critical t-value at $\alpha = 0.05$ (two-sided)
across different sample sizes:

In [ ]:
# qt() gives the quantile of the t-distribution — analogous to qnorm()
# For a two-sided test at alpha=0.05, we want the 97.5th percentile
sample_sizes = c(5, 10, 20, 30, 50, 100)
critical_t = qt(0.975, df = sample_sizes - 1)

data.frame(
  sample_size = sample_sizes,
  df = sample_sizes - 1,
  critical_t = round(critical_t, 3)
)

# For comparison: the Normal critical value is always 1.96
cat("\nNormal (z) critical value:", round(qnorm(0.975), 3), "\n")

With only 5 observations (df = 4), you need a t-score above 2.78 to reach
significance — considerably higher than the 1.96 threshold for a z-test.
As N grows, the t critical value converges toward 1.96.

---

## 2. The z-test by Hand

As covered in the slides, the z-test is a stepping stone to the t-test.
It requires knowing the **population standard deviation** ($\sigma_0$) —
an assumption the t-test relaxes.

**Example from the slides**: do linguistics students have the same average
grade as the overall class?

- $\mu_0 = 67.5$ (known class mean)
- $\sigma_0 = 9.5$ (known class standard deviation — the z-test assumption)
- $N = 20$ linguistics students
- $\bar{X} = 72.3$ (their observed mean grade)

In [ ]:
null_mean  = 67.5
null_stdev = 9.5   # assumed known population stdev
N          = 20
ling_mean  = 72.3

# Standard Error of the Mean
SEM = null_stdev / sqrt(N)
SEM

# z-score: how many SEMs away is our sample mean from the null mean?
z_score = (ling_mean - null_mean) / SEM
z_score

# One-sided p-value: P(Z >= z_score) under the null
upper_area = pnorm(z_score, lower.tail = FALSE)
upper_area

The p-value of 0.012 means: if the Null were true, there would be only a
1.2% chance of observing a sample mean as high as 72.3 (or higher).
That's significant at $\alpha = 0.05$.

But this relied on knowing $\sigma_0 = 9.5$ for the whole class — an
assumption we usually can't meet. That's why we use the t-test instead.

---

## 3. One-Sample t-test

The t-test makes one key change: instead of using the known population
standard deviation $\sigma_0$, it **estimates** it from the sample
($\hat{\sigma} = \sigma(X)$, the sample standard deviation).
The t-score formula is otherwise identical to the z-score:

$$t = \frac{\bar{X} - \mu_0}{\hat{\sigma} / \sqrt{N}}$$

The significance is then found from the **t-distribution** (not Normal),
using `pt()` instead of `pnorm()`.

### From Scratch

In [ ]:
null_mean  = 67.5
ling_mean  = 72.3
ling_stdev = 10.1  # estimated from the sample — note: slightly different from null_stdev
N          = 20

# SEM using the estimated stdev from the sample
SEM = ling_stdev / sqrt(N)
SEM

# t-score
t_score = (ling_mean - null_mean) / SEM
t_score

# One-sided p-value from the t-distribution with df = N - 1
upper_area = pt(t_score, df = N - 1, lower.tail = FALSE)
upper_area

The p-value is now 0.023 — slightly higher than the z-test's 0.012, because
the t-distribution has fatter tails (more uncertainty with a small sample).
Still significant at $\alpha = 0.05$.

### With `t.test()`

In practice, you won't calculate this by hand. R's `t.test()` does it all:

In [ ]:
# Simulate the linguistics grades dataset to match the slide example
set.seed(1)
ling_grades = rnorm(n = 20, mean = 72.3, sd = 10.1)

# One-sample t-test: is the mean different from 67.5?
t.test(x = ling_grades, mu = 67.5, alternative = "greater")

The output gives you the t-statistic, degrees of freedom, p-value, confidence
interval, and sample mean — everything you need to report the result.

### Applied to Vowel Data

A more linguistically relevant example: suppose you know from the literature
that the average F1 for the vowel /æ/ in American English is around 800 Hz.
Does the /æ/ in our dataset have the same mean F1?

In [ ]:
ae_vowels = filter(vowels, VOWEL == "æ")
cat("n =", nrow(ae_vowels), "| mean F1 =", round(mean(ae_vowels$F1), 1), "Hz\n")

t.test(x = ae_vowels$F1, mu = 800)

<details>
<summary><b>Challenge:</b> Run a one-sample t-test to check whether the mean
F1 for the vowel /i/ in our data differs from a literature value of 280 Hz.
Is the result significant at α = 0.05?</summary>

```r
i_vowels = filter(vowels, VOWEL == "i")
t.test(x = i_vowels$F1, mu = 280)
```

Check the p-value. If p < 0.05, the mean F1 for /i/ in our data is
significantly different from 280 Hz.
</details>

---

## 4. Independent Samples t-test

The **independent samples t-test** asks whether **two separate groups** have
the same population mean:

- $H_0$: $\mu_1 = \mu_2$
- $H_1$: $\mu_1 \neq \mu_2$

This is the most relevant variant for many linguistic research questions:
does group A produce different F1 values than group B?

The `t.test()` syntax for two samples is `t.test(x, y)` — or, for data frames,
`t.test(outcome ~ group, data = df)` (more on this formula notation when we
get to regression).

By default, R uses the **Welch t-test**, which does not assume equal variances
across groups. This is the safer choice in practice.

### Example: F1 by Sex

Do male and female speakers in our dataset have different mean F1 values
overall?

In [ ]:
f1_male   = filter(vowels, SEX == "male")$F1
f1_female = filter(vowels, SEX == "female")$F1

cat("Male   mean F1:", round(mean(f1_male), 1), "Hz (n =", length(f1_male), ")\n")
cat("Female mean F1:", round(mean(f1_female), 1), "Hz (n =", length(f1_female), ")\n")

In [ ]:
# Welch two-sample t-test (the default)
t.test(f1_female, f1_male)

### Visualizing the Comparison

It's always good practice to plot the data alongside the test result.

In [ ]:
boxplot(F1 ~ SEX, data = vowels,
        xlab = "Sex", ylab = "F1 (Hz)",
        main = "F1 Distribution by Speaker Sex")

### Example: Comparing Two Vowels

Do /i/ and /u/ have different mean F1 values? These are both high vowels,
but differ in backness.

In [ ]:
f1_i = filter(vowels, VOWEL == "i")$F1
f1_u = filter(vowels, VOWEL == "u")$F1

cat("/i/ mean F1:", round(mean(f1_i), 1), "Hz\n")
cat("/u/ mean F1:", round(mean(f1_u), 1), "Hz\n")

t.test(f1_i, f1_u)

<details>
<summary><b>Challenge:</b> Test whether /æ/ and /e/ have significantly
different mean F1 values. What do you expect, and what does the test
say?</summary>

```r
f1_ae = filter(vowels, VOWEL == "æ")$F1
f1_e  = filter(vowels, VOWEL == "e")$F1
cat("/æ/ mean F1:", round(mean(f1_ae), 1), "Hz\n")
cat("/e/  mean F1:", round(mean(f1_e), 1), "Hz\n")
t.test(f1_ae, f1_e)
```

/æ/ is a low front vowel and /e/ is a mid front vowel, so we'd expect /æ/
to have higher F1 (lower tongue position = higher F1). Check whether
that difference is statistically significant.
</details>

### Student's vs. Welch t-test

The **Student's t-test** additionally assumes the two groups have the **same
variance**. You can request it with `var.equal = TRUE`, but unless you have
strong reason to believe the variances are equal, the Welch test (the default)
is the safer choice.

In [ ]:
# Check the standard deviations before deciding
cat("SD F1 female:", round(sd(f1_female), 1), "Hz\n")
cat("SD F1 male:  ", round(sd(f1_male), 1), "Hz\n")

# Student's t-test (equal variance assumed)
t.test(f1_female, f1_male, var.equal = TRUE)

For this data, the two tests give very similar results. When sample sizes
and variances are similar, Student's and Welch usually agree — but Welch is
more robust when they differ.

---

## Quick Reference

| Task | R code |
|------|--------|
| t-distribution density | `dt(x, df)` |
| Cumulative t-distribution | `pt(q, df, lower.tail = FALSE)` |
| t critical value | `qt(0.975, df)` |
| One-sample t-test | `t.test(x, mu = null_mean)` |
| One-sided one-sample t-test | `t.test(x, mu = null_mean, alternative = "greater")` |
| Two-sample Welch t-test | `t.test(x, y)` |
| Two-sample Student's t-test | `t.test(x, y, var.equal = TRUE)` |
| Formula syntax | `t.test(outcome ~ group, data = df)` |

| | One-sample | Independent samples |
|---|---|---|
| **Question** | Does this sample come from a population with mean $\mu_0$? | Do these two samples come from populations with the same mean? |
| **$H_0$** | $\mu = \mu_0$ | $\mu_1 = \mu_2$ |
| **Stdev used** | Estimated from sample | Estimated from each sample |
| **R function** | `t.test(x, mu = ...)` | `t.test(x, y)` |